Working Plan
1. Install libraries
2. Import libraries
3. Load lightweight LLM
4. Test model
5. Create answering function
6. Build Gradio interface
7. Test with sample reports
8. Save screenshots for submission

Install libraries

In [1]:
!pip install transformers gradio sentencepiece torch

In [11]:
!pip uninstall -y transformers
!pip install transformers==4.41.2 sentencepiece accelerate torch gradio

Found existing installation: transformers 4.41.2
Uninstalling transformers-4.41.2:
  Successfully uninstalled transformers-4.41.2
  Using cached transformers-4.41.2-py3-none-any.whl.metadata (43 kB)
Using cached transformers-4.41.2-py3-none-any.whl (9.1 MB)


Import libraries

In [1]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, pipeline
import gradio as gr

In [2]:
model_name = "google/flan-t5-small"

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

nlp_pipeline = pipeline(
    task="text2text-generation",
    model=model,
    tokenizer=tokenizer
)

print("Model loaded successfully.")



/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Model loaded successfully.


transformers → loads the AI model
gradio → creates the web app

This is the “AI brain” of your chatbot.

Test the model

In [3]:
test_prompt = """
question: What is the diagnosis?
context: The patient presents with persistent cough and fever. Chest X-ray shows signs of pneumonia.
"""

result = nlp_pipeline(test_prompt)

print(result[0]["generated_text"])

pneumonia


/usr/local/lib/python3.12/dist-packages/transformers/generation/utils.py:1168: UserWarning: Using the model-agnostic default `max_length` (=20) to control the generation length. We recommend setting `max_new_tokens` to control the maximum length of the generation.
  warnings.warn(


Create the answering function

Create the answering function

In [4]:
def generate_answer(report, question):
    if not report or not question:
        return "Please enter both a medical report and a question."

    prompt = f"question: {question} context: {report}"

    result = nlp_pipeline(
        prompt,
        max_new_tokens=100
    )

    answer = result[0]["generated_text"]

    return answer

Test the function without Gradio

In [5]:
sample_report = """
The patient presents with persistent cough and fever.
Chest X-ray shows signs of pneumonia.
Antibiotics have been prescribed.
"""

sample_question = "What is the diagnosis?"

answer = generate_answer(sample_report, sample_question)

print("Question:", sample_question)
print("Answer:", answer)

Question: What is the diagnosis?
Answer: pneumonia


Create the Gradio interface

In [6]:
report_input = gr.Textbox(
    lines=10,
    label="Enter Medical Report",
    placeholder="Paste a clinical note, discharge summary, lab report, or radiology report here..."
)

question_input = gr.Textbox(
    lines=2,
    label="Enter Your Question",
    placeholder="Example: What is the diagnosis?"
)

answer_output = gr.Textbox(
    lines=4,
    label="AI Answer"
)

iface = gr.Interface(
    fn=generate_answer,
    inputs=[report_input, question_input],
    outputs=answer_output,
    title="Generative AI Medical Chatbot",
    description="Enter a medical report and ask a clinical question. This prototype uses a lightweight language model to generate an educational answer.",
    submit_btn="Get the Answer"
)

iface.launch()

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://ed9090cd9af9709b99.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [7]:
iface.close()

Closing server running on port: 7860


Test example 2